# Intro 10 — Confidence Intervals for a Mean

Practice notebook for the **"Confidence Intervals for a Mean"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Known variance: the z-interval

Suppose \(X_1,\dots,X_n\) are i.i.d. \(N(\mu,\sigma^2)\) with \(\sigma^2\) known.
Then \(\bar X \sim N(\mu, \sigma^2/n)\). A \((1-\alpha)\)-level confidence interval for \(\mu\) is

$$
\bar X \pm z_{\alpha/2}\,\sigma/\sqrt{n},
$$

where \(z_{\alpha/2}\) is the upper \(\alpha/2\) quantile of the standard normal
(e.g. \(z_{0.025}\approx 1.96\) for 95% confidence).

Interpretation: over many repetitions, a fraction \(1-\alpha\) of such intervals
contain the true mean \(\mu\).

PDF example: known \(\sigma=2\)%, \(n=100\), \(\bar X=0.3\)%. The 95% CI is

$$
0.3\% \pm 1.96\cdot 2\%/\sqrt{100}
= 0.3\% \pm 0.392\%
\approx [-0.092\%,\, 0.692\%].
$$

We reproduce it from scratch and with scipy.stats.


In [ ]:
# PDF example: known sigma = 2%, n = 100, xbar = 0.3%, 95% CI
sigma = 2.0
n = 100
xbar = 0.3
alpha = 0.05

z_half = stats.norm.ppf(1 - alpha/2)
se = sigma/np.sqrt(n)
lo, hi = xbar - z_half*se, xbar + z_half*se
print('z_{0.025} =', round(z_half, 6), '(expected 1.959964)')
print('95% CI (scratch): [', round(lo, 6), ',', round(hi, 6), '] %')

lo2, hi2 = stats.norm.interval(1 - alpha, loc=xbar, scale=se)
print('95% CI (scipy):   [', round(lo2, 6), ',', round(hi2, 6), '] %')

assert np.isclose(z_half, 1.959963984540054, atol=1e-6)
assert np.isclose(lo, -0.092, atol=1e-3) and np.isclose(hi, 0.692, atol=1e-3)
print('Matches PDF example OK')


---
## Part 2 — Unknown variance: the t-interval

If \(\sigma^2\) is unknown, replace it with the sample sd \(s\). For normal data,
\(T = (\bar X - \mu)/(s/\sqrt{n})\) has a Student's t distribution with \(n-1\)
degrees of freedom. A \((1-\alpha)\)-level CI for \(\mu\) is

$$
\bar X \pm t_{\alpha/2,\,n-1}\,s/\sqrt{n}.
$$

PDF example: execution time (ms), \(n=20\), \(\bar X=105\), \(s=10\), \(n-1=19\)
d.f., \(t_{0.025,19}\approx 2.093\). The 95% CI is

$$
105 \pm 2.093\cdot 10/\sqrt{20}\approx 105 \pm 4.68\approx [100.3,\, 109.7].
$$

For large \(n\), \(t_{\alpha/2,\,n-1}\to z_{\alpha/2}\); for small \(n\) the
t-interval is wider because it accounts for the extra uncertainty in estimating
\(\sigma\) with \(s\).


In [ ]:
# PDF example: n=20, xbar=105, s=10, 95% CI, 19 d.f.
n = 20
xbar = 105.0
s = 10.0
alpha = 0.05
df = n - 1

t_half = stats.t.ppf(1 - alpha/2, df)
se = s/np.sqrt(n)
lo, hi = xbar - t_half*se, xbar + t_half*se
print('t_{0.025,19} =', round(t_half, 6), '(expected ~2.093024)')
print('95% CI (scratch): [', round(lo, 4), ',', round(hi, 4), '] ms')

lo2, hi2 = stats.t.interval(1 - alpha, df, loc=xbar, scale=se)
print('95% CI (scipy):   [', round(lo2, 4), ',', round(hi2, 4), '] ms')

z_half = stats.norm.ppf(1 - alpha/2)
print()
print('   n |  t_half   z_half   t/z')
for nn in [5, 10, 20, 50, 200]:
    th = stats.t.ppf(1 - alpha/2, nn - 1)
    print(f'{nn:4d} | {th:.4f}  {z_half:.4f}  {th/z_half:.4f}')

assert np.isclose(t_half, 2.09302405435263, atol=1e-5)
assert np.isclose(lo, 100.3, atol=1e-1) and np.isclose(hi, 109.7, atol=1e-1)
print()
print('Matches PDF example OK')


---
## Part 3 — Coverage frequency: do CIs contain \(\mu\) at the nominal rate?

A \((1-\alpha)\) CI is defined by frequency: over many repetitions, a fraction
\(1-\alpha\) of the intervals contain \(\mu\). We draw \(B=20000\) samples of size
\(n\) from \(N(\mu,\sigma^2)\) and build three intervals:

- z-CI: uses the known \(\sigma\) — coverage \(\approx 1-\alpha\).
- t-CI: uses the sample \(s\) — coverage \(\approx 1-\alpha\) for normal data.
- naive z-CI that plugs in \(s\) as if it were \(\sigma\): under-covers for small \(n\).


In [ ]:
# Coverage simulation: build many CIs, count how many contain mu
mu_true = 3.0
sigma_true = 2.0
alpha = 0.05
B = 20000

def simulate_coverage(n, B=B, seed=42):
    rng = np.random.default_rng(seed)
    samples = rng.normal(mu_true, sigma_true, size=(B, n))
    xbar = samples.mean(axis=1)
    s = samples.std(axis=1, ddof=1)
    se_z = sigma_true/np.sqrt(n)
    se_t = s/np.sqrt(n)
    z_half = stats.norm.ppf(1 - alpha/2)
    t_half = stats.t.ppf(1 - alpha/2, n - 1)
    cov_z = ((xbar - z_half*se_z <= mu_true) & (mu_true <= xbar + z_half*se_z)).mean()
    cov_t = ((xbar - t_half*se_t <= mu_true) & (mu_true <= xbar + t_half*se_t)).mean()
    cov_naive = ((xbar - z_half*se_t <= mu_true) & (mu_true <= xbar + z_half*se_t)).mean()
    return cov_z, cov_t, cov_naive

ns = [5, 10, 20, 50, 100, 500]
print('   n |   z-CI    t-CI   naive')
for n in ns:
    cz, ct, cn = simulate_coverage(n)
    print(f'{n:4d} | {cz:.4f}  {ct:.4f}  {cn:.4f}')

print()
print('Nominal level: 0.95')
print('z-CI and t-CI match nominal; naive z-CI under-covers for small n.')

cz500, ct500, _ = simulate_coverage(500)
assert abs(cz500 - 0.95) < 0.02 and abs(ct500 - 0.95) < 0.02
print('Large-n coverage OK')


---
**Your turn:**

- Re-run Part 3 with a non-normal population (e.g. an exponential). Do the z-CI
  and t-CI still hit 95% for small \(n\)? Why or why not? (Think CLT.)
- Change the confidence level to 99% and 90%. How do the intervals' width and
  coverage change?
- Increase B to 100000. Does the coverage frequency get closer to 0.95? Why does
  it take many simulations for the frequency interpretation to become visible?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. A sample of size \(n=50\) from a normal population has \(\bar X=12.4\) and
known \(\sigma=1.5\). Build a 99% z-CI for \(\mu\) from scratch (compute
\(z_{0.005}\) with `scipy.stats.norm.ppf`).

2. A sample of size \(n=12\) from a normal population has \(\bar X=8.2\) and
sample sd \(s=2.0\). Build a 95% t-CI for \(\mu\). Report the degrees of freedom,
\(t_{0.025,11}\), and the interval endpoints.

3. Simulate \(B=10000\) samples of size \(n=15\) from \(N(5, 3^2)\) with seed
0. For each, build a 95% t-CI. Report the empirical coverage and the mean
interval width. Confirm coverage is \(\approx 0.95\).

4. Show that for a 95% level and \(n=8\), the naive plug-in z-CI (plug in
\(s\) for \(\sigma\) but use \(z_{0.025}\) instead of \(t_{0.025,7}\))
under-covers. Report its simulated coverage.

5. Write a function `ci_mean(x, level=0.95, sigma=None)` that returns
`(lo, hi)` using the z-interval when `sigma` is given and the t-interval
otherwise. Test it on the PDF known-variance example (returns
\([-0.092, 0.692]\)) and on the PDF t example (returns \([100.3, 109.7]\)).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(z_{0.005} = \Phi^{-1}(0.995) \approx 2.5758\). The
standard error is \(\sigma/\sqrt{n} = 1.5/\sqrt{50} \approx 0.2121\). The
99% CI is \(12.4 \pm 2.5758 \cdot 0.2121 \approx [11.854,\, 12.946]\).

```python
n, xbar, sigma, alpha = 50, 12.4, 1.5, 0.01
z = stats.norm.ppf(1 - alpha/2)
se = sigma/np.sqrt(n)
print(xbar - z*se, xbar + z*se)  # 11.8543 12.9457
```

**2.** d.f. \(= n-1 = 11\); \(t_{0.025, 11} \approx 2.2010\). The standard
error is \(s/\sqrt{n} = 2.0/\sqrt{12} \approx 0.5774\). The 95% CI is
\(8.2 \pm 2.2010 \cdot 0.5774 \approx [6.929,\, 9.471]\).

```python
n, xbar, s, alpha = 12, 8.2, 2.0, 0.05
df = n - 1
t = stats.t.ppf(1 - alpha/2, df)
se = s/np.sqrt(n)
print(t, xbar - t*se, xbar + t*se)  # 2.2011 6.9292 9.4708
```

**3.** Coverage should be \(\approx 0.95\); the mean width is
\(2 \cdot t_{0.025,14} \cdot s/\sqrt{n}\) averaged over simulations.

```python
rng = np.random.default_rng(0)
n, B, mu, sigma, alpha = 15, 10_000, 5.0, 3.0, 0.05
X = rng.normal(mu, sigma, size=(B, n))
xbar, s = X.mean(1), X.std(1, ddof=1)
t = stats.t.ppf(1 - alpha/2, n - 1)
se = s/np.sqrt(n)
lo, hi = xbar - t*se, xbar + t*se
cov = ((lo <= mu) & (mu <= hi)).mean()
width = (hi - lo).mean()
print(cov, width)  # ~0.949, ~3.31
```

**4.** With \(n=8\), \(z_{0.025}=1.96\) but \(t_{0.025,7}\approx 2.3646\).
Ignoring the extra uncertainty shrinks the interval, so coverage drops
below 0.95 (typically \(\approx 0.90\) for \(n=8\)).

```python
rng = np.random.default_rng(1)
n, B, mu, sigma, alpha = 8, 20_000, 0.0, 1.0, 0.05
X = rng.normal(mu, sigma, size=(B, n))
xbar, s = X.mean(1), X.std(1, ddof=1)
z = stats.norm.ppf(1 - alpha/2)
se = s/np.sqrt(n)
covers = ((xbar - z*se <= mu) & (mu <= xbar + z*se)).mean()
print(covers)  # ~0.90, under-covering
```

**5.** A reusable CI function. For the exact PDF t example, plug the summary
statistics into the t formula directly (as in Part 2), which gives
\([100.3, 109.7]\).

```python
def ci_mean(x, level=0.95, sigma=None):
    x = np.asarray(x, dtype=float)
    n = x.size
    xbar = x.mean()
    alpha = 1 - level
    if sigma is not None:
        crit = stats.norm.ppf(1 - alpha/2)
        se = sigma / np.sqrt(n)
    else:
        crit = stats.t.ppf(1 - alpha/2, n - 1)
        se = x.std(ddof=1) / np.sqrt(n)
    return xbar - crit*se, xbar + crit*se
```


</details>
